# AnuraSetEurope: data aquisition

This notebook (re)builds the European anuran audio dataset described in [Zenodo](https://doi.org/10.5281/zenodo.21282595). It is designed to be rerun with the most current data available from the source repositories, rather than reproducing the historical (15 July 2026) snapshot exactly.

The steps are as follows:

- Define the target taxonomic scope (Europe).
- Retrieve metadata and sound URLs from Xeno-canto, iNaturalist, and Observation.org/GBIF.
- Download audio.

**Important difference from the manuscript snapshot:** every time this notebook is rerun, the public repositories may contain additional records, edited records, changed taxonomy, changed licences, or removed media. The output is therefore a new dated snapshot.  

In the second notebook, [AnuraSetEurope_preprocessing.ipynb](https://doi.org/10.5281/zenodo.21282595), we will work on cleaning the retrieved data.

*Notebook author: Jagoda Hanuszewicz*

## 0. Installation

Run this once in a fresh environment. `ffmpeg` is strongly recommended because public audio files may be MP3, M4A, WAV, or OGG.

On Linux, install ffmpeg first if needed:

```bash
sudo apt-get update && sudo apt-get install -y ffmpeg
```

In [ ]:
# Uncomment in a fresh notebook environment.
# %pip install -q pandas numpy requests tqdm librosa soundfile pydub mutagen matplotlib pycountry unidecode packaging

# Optional, only needed for map-style country coverage figures.
# %pip install -q geopandas geodatasets shapely

## 1. Imports and configuration

Edit only this cell before running the notebook.

You need a Xeno-canto API key for the current API. Put it in an environment variable before launching Jupyter:

```bash
export XENO_CANTO_API_KEY="your_key_here"
```

The iNaturalist and GBIF/Observation.org sections use public endpoints and do not normally require credentials.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# Audio packages are imported later so the metadata-only cells can still run.

RUN_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# Project folders
ROOT = Path("AnuraSetEurope_rebuild")
RAW_DIR = ROOT / "raw"
RAW_META_DIR = RAW_DIR / "metadata"
RAW_AUDIO_DIR = RAW_DIR / "audio"
WORK_DIR = ROOT / "work"
CURATED_AUDIO_TMP = WORK_DIR / "curated_wav_unassigned"
OUT_META_DIR = ROOT / "metadata"
OUT_AUDIO_DIR = ROOT / "audio"
OUT_FIG_DIR = ROOT / "figures"
OUT_TABLE_DIR = ROOT / "tables"

for d in [RAW_META_DIR, RAW_AUDIO_DIR, WORK_DIR, CURATED_AUDIO_TMP, OUT_META_DIR, OUT_AUDIO_DIR, OUT_FIG_DIR, OUT_TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Main curation parameters from the manuscript methods.
TARGET_SAMPLE_RATE = 32_000
MIN_DURATION_SEC = 2.0
MAX_DURATION_SEC = 300.0
SPLIT_RATIOS = {"train": 0.60, "validation": 0.20, "test": 0.20}

# Set to a small number, e.g. 2, while testing the notebook structure.
# Set to None for the real rebuild.
MAX_RECORDS_PER_SOURCE_SPECIES: Optional[int] = None

# Metadata-only mode is useful while checking API retrieval before downloading audio.
DOWNLOAD_AUDIO = True
PROCESS_AUDIO = True

# Be polite to public APIs and media servers.
REQUEST_SLEEP_SEC = 0.25
DOWNLOAD_SLEEP_SEC = 0.15
REQUEST_TIMEOUT = 60

# Use your own contact if submitting many requests.
USER_AGENT = "AnuraSetEurope-rebuild/1.0 (academic dataset rebuild; contact: replace_with_email@example.org)"
HEADERS = {"User-Agent": USER_AGENT}

# Current iNaturalist Europe place identifier used in the manuscript draft.
INATURALIST_EUROPE_PLACE_ID = 97391

# Observation.org dataset on GBIF.
# Source: GBIF dataset page: "Observation.org, Nature data from around the World".
GBIF_OBSORG_DATASET_KEY = "8a863029-f435-446a-821e-275f4f641165"

# GBIF and Xeno-canto use country filters differently. We keep a broad ISO2 list for Europe.
# This list includes several transcontinental/candidate countries used in European biodiversity workflows.
EUROPE_COUNTRY_CODES = sorted(set("""
AL AD AT BY BE BA BG HR CY CZ DK EE FI FR DE GI GR HU IS IE IT XK LV LI LT LU MT MD MC ME NL MK NO PL PT RO RU SM RS SK SI ES SE CH TR UA GB VA
""".split()))

# Serial-recording throttling. Including latin avoids discarding different species recorded by the same observer
# at the same place and hour. If you need to match the manuscript wording strictly, remove "latin".
SERIAL_THROTTLE_COLUMNS = ["latin", "recordist_norm", "lat_round_5", "lon_round_5", "date", "hour"]

print(f"Run date: {RUN_DATE}")
print(f"Output root: {ROOT.resolve()}")

Run date: 2026-07-15
Output root: /data/jagoda/publication/AnuraSetEurope_rebuild


## 2. Target taxonomy

We are interested in recordings of anurans made in Europe.

In [ ]:
DISCOVERY_SCOPE = "Anura"
DISCOVERY_QUERY_NAME = "all European amphibian sound records"

# Keep this True if you want a species-level ML dataset.
# It drops records that are only identified to genus/family/class level.
INCLUDE_ONLY_BINOMIAL_SPECIES = True

# Used only for Xeno-canto post-filtering, because iNaturalist and GBIF
# already support Europe/place/continent filters.
EUROPE_COUNTRY_NAMES_XC = {
    "Albania", "Andorra", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina",
    "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia",
    "Finland", "France", "Germany", "Gibraltar", "Greece", "Hungary", "Iceland",
    "Ireland", "Italy", "Kosovo", "Latvia", "Liechtenstein", "Lithuania",
    "Luxembourg", "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands",
    "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "Russia",
    "San Marino", "Serbia", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Switzerland", "Turkey", "Ukraine", "United Kingdom", "Vatican City"
}

,scope,query,include_only_binomial_species
0,Anura,all European amphibian sound records,True


## 3. Utility functions

In [3]:
def request_json(url: str, params: Optional[dict] = None, *, retries: int = 3, sleep: float = REQUEST_SLEEP_SEC) -> dict:
    """Small wrapper around requests.get() with retries and JSON parsing."""
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if r.status_code == 429:
                time.sleep(max(5, sleep * 10))
                continue
            r.raise_for_status()
            time.sleep(sleep)
            return r.json()
        except Exception as exc:
            last_error = exc
            time.sleep(sleep * attempt)
    raise RuntimeError(f"Failed request: {url} params={params} error={last_error}")


def normalise_text(x: Any) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x


def slugify(x: str) -> str:
    x = normalise_text(x)
    x = re.sub(r"[^A-Za-z0-9_. -]+", "", x)
    return x.replace(" ", "_") or "unknown"


def safe_float(x: Any) -> float:
    try:
        if x in [None, "", "null"]:
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def safe_filename_from_url(url: str, default: str) -> str:
    path = urlparse(str(url)).path
    name = Path(path).name
    if not name or "." not in name:
        name = default
    return slugify(name)


def md5_string(x: str) -> str:
    return hashlib.md5(x.encode("utf-8")).hexdigest()


def split_from_group_key(group_key: str) -> str:
    """Stable 60/20/20 split from an MD5 hash of the acoustic-session group key."""
    h = int(hashlib.md5(group_key.encode("utf-8")).hexdigest(), 16) / 2**128
    if h < SPLIT_RATIOS["train"]:
        return "train"
    if h < SPLIT_RATIOS["train"] + SPLIT_RATIOS["validation"]:
        return "validation"
    return "test"


def parse_date_time(value: Any) -> Tuple[str, str, str]:
    """Return date, time, hour strings from a flexible date/time value."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "", "", ""
    txt = str(value)
    dt = pd.to_datetime(txt, errors="coerce", utc=False)
    if pd.isna(dt):
        return "", "", ""
    date = dt.strftime("%Y-%m-%d")
    time_s = dt.strftime("%H:%M:%S") if not pd.isna(dt) else ""
    hour = dt.strftime("%H") if not pd.isna(dt) else ""
    return date, time_s, hour


def choose_first(*values: Any) -> Any:
    for v in values:
        if v is not None and not (isinstance(v, float) and math.isnan(v)) and str(v) != "":
            return v
    return None


def download_binary(url: str, out_path: Path, *, overwrite: bool = False) -> bool:
    """Download one media file. Returns True if file exists after the call."""
    if not url or not isinstance(url, str):
        return False
    if out_path.exists() and out_path.stat().st_size > 0 and not overwrite:
        return True
    out_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, stream=True, allow_redirects=True) as r:
            r.raise_for_status()
            tmp = out_path.with_suffix(out_path.suffix + ".part")
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
            tmp.replace(out_path)
        time.sleep(DOWNLOAD_SLEEP_SEC)
        return out_path.exists() and out_path.stat().st_size > 0
    except Exception as exc:
        print(f"Download failed: {url} -> {out_path} ({exc})")
        return False

## 4. Retrieve source metadata

The notebook saves raw source metadata before downloading any audio. This gives you a clear checkpoint and makes failed audio downloads easier to diagnose.

### 4.1 Xeno-canto

Current Xeno-canto API access may require an API key. The code first tries the current parameter-style API. If your local Xeno-canto query syntax differs, edit `xc_api_query()` only.

In [ ]:
def normalise_country_for_filter(x: Any) -> str:
    return normalise_text(x).lower()


EUROPE_COUNTRY_NAMES_XC_NORM = {
    normalise_country_for_filter(x) for x in EUROPE_COUNTRY_NAMES_XC
}


def xc_api_query_all_amphibians() -> List[dict]:
    api_key = "3e694db5cc19041593c97584a113d98657f1a37e" # Jagoda's key; replace before sumbitting!!!
    endpoint = "https://xeno-canto.org/api/3/recordings"
    records = []
    page = 1

    while True:
        # Try parameter-style first, then query-string style.
        try:
            data = request_json(endpoint, params={
                "key": api_key,
                "grp": "frogs",
                "page": page,
            })
        except Exception:
            data = request_json(endpoint, params={
                "key": api_key,
                "query": "grp:frogs",
                "page": page,
            })

        page_records = data.get("recordings", []) or []
        records.extend(page_records)

        num_pages = int(data.get("numPages") or data.get("num_pages") or 1)
        if page >= num_pages:
            break
        page += 1

    return records


def normalise_xc_record(rec: dict) -> dict:
    source_id_raw = choose_first(rec.get("id"), rec.get("xcid"), rec.get("nr"))
    source_id = f"XC{source_id_raw}" if source_id_raw is not None and not str(source_id_raw).startswith("XC") else str(source_id_raw)

    audio_url = choose_first(rec.get("file"), rec.get("file-name"), rec.get("download"))
    date = normalise_text(rec.get("date"))
    time_s = normalise_text(rec.get("time"))

    if date and time_s:
        parsed_date, parsed_time, hour = parse_date_time(f"{date} {time_s}")
    else:
        parsed_date, parsed_time, hour = parse_date_time(date)

    repository_latin = normalise_text(
        " ".join([normalise_text(rec.get("gen")), normalise_text(rec.get("sp"))])
    ).strip()

    return {
        "source": "xc",
        "source_id": source_id,
        "source_record_id": source_id_raw,
        "sound_id": source_id_raw,
        "query_name": DISCOVERY_SCOPE,
        "repository_taxon_id": "",
        "latin": repository_latin,
        "repository_latin": repository_latin,
        "common_name": choose_first(rec.get("en"), rec.get("common_name"), ""),
        "recordist": choose_first(rec.get("rec"), rec.get("recordist")),
        "observer": choose_first(rec.get("rec"), rec.get("recordist")),
        "date": parsed_date,
        "time": parsed_time or time_s,
        "hour": hour,
        "when_uploaded": choose_first(rec.get("uploaded"), rec.get("uploaded_date")),
        "country": choose_first(rec.get("cnt"), rec.get("country")),
        "country_code": "",
        "location": choose_first(rec.get("loc"), rec.get("location")),
        "latitude": safe_float(choose_first(rec.get("lat"), rec.get("latitude"))),
        "longitude": safe_float(choose_first(rec.get("lng"), rec.get("lon"), rec.get("longitude"))),
        "remarks": choose_first(rec.get("rmk"), rec.get("remarks"), rec.get("notes")),
        "license": choose_first(rec.get("lic"), rec.get("license")),
        "audio_quality": choose_first(rec.get("q"), rec.get("quality")),
        "verification": "xeno-canto",
        "audio_url": audio_url,
        "observation_url": choose_first(rec.get("url"), f"https://xeno-canto.org/{source_id_raw}" if source_id_raw else None),
        "file_name": safe_filename_from_url(str(audio_url), f"{source_id}.audio"),
        "raw_json": json.dumps(rec, ensure_ascii=False),
    }


def fetch_xenocanto_metadata() -> pd.DataFrame:
    records = xc_api_query_all_amphibians()

    if MAX_RECORDS_PER_SOURCE_SPECIES:
        records = records[:MAX_RECORDS_PER_SOURCE_SPECIES]

    rows = []
    seen_ids = set()

    for rec in records:
        row = normalise_xc_record(rec)
        country_norm = normalise_country_for_filter(row["country"])

        if country_norm not in EUROPE_COUNTRY_NAMES_XC_NORM:
            continue

        if row["source_id"] not in seen_ids and row["audio_url"]:
            rows.append(row)
            seen_ids.add(row["source_id"])

    df = pd.DataFrame(rows)
    out = RAW_META_DIR / "xenocanto_raw_metadata.csv"
    df.to_csv(out, index=False)
    print(f"Saved {len(df):,} Xeno-canto metadata rows to {out}")
    return df


xc_meta = fetch_xenocanto_metadata()
xc_meta.head()

Saved 2,227 Xeno-canto metadata rows to AnuraSetEurope_rebuild/raw/metadata/xenocanto_raw_metadata.csv


,source,source_id,source_record_id,sound_id,query_name,repository_taxon_id,latin,repository_latin,common_name,recordist,...,latitude,longitude,remarks,license,audio_quality,verification,audio_url,observation_url,file_name,raw_json
0,xc,XC1155635,1155635,1155635,Anura,,Pelophylax bedriagae,Pelophylax bedriagae,Levante water frog,Jagoda Hanuszewicz,...,40.0900,29.5600,"Big chorus, big lake, many individuals.",https://creativecommons.org/licenses/by-nc-sa/...,A,xeno-canto,https://xeno-canto.org/1155635/download,https://xeno-canto.org/1155635,XC1155635.audio,"{""id"": ""1155635"", ""gen"": ""Pelophylax"", ""sp"": ""..."
1,xc,XC1155631,1155631,1155631,Anura,,Pelophylax bedriagae,Pelophylax bedriagae,Levante water frog,Jagoda Hanuszewicz,...,39.5152,29.3816,"Small lake, camping site.",https://creativecommons.org/licenses/by-nc-sa/...,A,xeno-canto,https://xeno-canto.org/1155631/download,https://xeno-canto.org/1155631,XC1155631.audio,"{""id"": ""1155631"", ""gen"": ""Pelophylax"", ""sp"": ""..."
2,xc,XC1155624,1155624,1155624,Anura,,Pelophylax bedriagae,Pelophylax bedriagae,Levante water frog,Jagoda Hanuszewicz,...,39.4100,30.1900,None,https://creativecommons.org/licenses/by-nc-sa/...,A,xeno-canto,https://xeno-canto.org/1155624/download,https://xeno-canto.org/1155624,XC1155624.audio,"{""id"": ""1155624"", ""gen"": ""Pelophylax"", ""sp"": ""..."
3,xc,XC1155621,1155621,1155621,Anura,,Pelophylax bedriagae,Pelophylax bedriagae,Levante water frog,Jagoda Hanuszewicz,...,40.2100,32.2500,Recorded near a small and slow river.,https://creativecommons.org/licenses/by-nc-sa/...,A,xeno-canto,https://xeno-canto.org/1155621/download,https://xeno-canto.org/1155621,XC1155621.audio,"{""id"": ""1155621"", ""gen"": ""Pelophylax"", ""sp"": ""..."
4,xc,XC1155620,1155620,1155620,Anura,,Pelophylax bedriagae,Pelophylax bedriagae,Levante water frog,Jagoda Hanuszewicz,...,40.2100,32.2500,Recorded near small and slow river.,https://creativecommons.org/licenses/by-nc-sa/...,A,xeno-canto,https://xeno-canto.org/1155620/download,https://xeno-canto.org/1155620,XC1155620.audio,"{""id"": ""1155620"", ""gen"": ""Pelophylax"", ""sp"": ""..."


### 4.2 iNaturalist

The iNaturalist query uses research-grade observations with sound files inside the Europe place (`place_id=97391`). The code resolves each current taxon ID first, then retrieves observations with `sounds=true`.

In [9]:
def inat_resolve_taxon_id(query_name: str, rank: Optional[str] = None) -> Optional[int]:
    url = "https://api.inaturalist.org/v1/taxa/autocomplete"
    params = {"q": query_name, "per_page": 10}
    if rank:
        params["rank"] = rank

    data = request_json(url, params=params)
    results = data.get("results", [])
    if not results:
        return None

    q_norm = query_name.lower()

    for r in results:
        name = normalise_text(r.get("name")).lower()
        matched = normalise_text(r.get("matched_term")).lower()
        result_rank = normalise_text(r.get("rank")).lower()
        if (name == q_norm or matched == q_norm) and (rank is None or result_rank == rank.lower()):
            return int(r["id"])

    for r in results:
        if r.get("is_active", True):
            return int(r["id"])

    return int(results[0]["id"])


def inat_fetch_amphibia_observations() -> Tuple[List[dict], int]:
    amphibia_taxon_id = inat_resolve_taxon_id("Amphibia", rank="class")
    if not amphibia_taxon_id:
        raise ValueError("Could not resolve iNaturalist taxon ID for Amphibia.")

    url = "https://api.inaturalist.org/v1/observations"
    page = 1
    records = []

    while True:
        params = {
            "taxon_id": amphibia_taxon_id,
            "place_id": INATURALIST_EUROPE_PLACE_ID,
            "quality_grade": "research",
            "sounds": "true",
            "per_page": 200,
            "page": page,
            "order_by": "id",
            "order": "asc",
            "locale": "en",
        }

        data = request_json(url, params=params)
        results = data.get("results", [])
        records.extend(results)

        total = int(data.get("total_results", len(records)))
        if len(records) >= total or not results:
            break
        page += 1

    return records, amphibia_taxon_id


def normalise_inat_sound(obs: dict, sound: dict, amphibia_taxon_id: int) -> dict:
    obs_id = obs.get("id")
    sound_id = choose_first(sound.get("id"), sound.get("uuid"), sound.get("native_sound_id"))
    source_id = f"INAT{obs_id}_{sound_id}" if sound_id else f"INAT{obs_id}"

    observed_on = choose_first(obs.get("observed_on"), obs.get("time_observed_at"), obs.get("observed_on_string"))
    date, time_s, hour = parse_date_time(observed_on)

    geojson = obs.get("geojson") or {}
    coords = geojson.get("coordinates") or [np.nan, np.nan]
    lon = coords[0] if len(coords) > 0 else np.nan
    lat = coords[1] if len(coords) > 1 else np.nan

    taxon = obs.get("taxon") or {}
    user = obs.get("user") or {}
    place_guess = choose_first(obs.get("place_guess"), obs.get("location"))
    audio_url = choose_first(sound.get("file_url"), sound.get("url"), sound.get("original_url"), sound.get("medium_url"))

    repository_latin = choose_first(taxon.get("name"), "")
    common_name = choose_first(taxon.get("preferred_common_name"), "")

    return {
        "source": "inat",
        "source_id": source_id,
        "source_record_id": obs_id,
        "sound_id": sound_id,
        "query_name": DISCOVERY_SCOPE,
        "repository_taxon_id": choose_first(taxon.get("id"), amphibia_taxon_id),
        "latin": repository_latin,
        "repository_latin": repository_latin,
        "common_name": common_name,
        "recordist": choose_first(user.get("login"), user.get("name")),
        "observer": choose_first(user.get("login"), user.get("name")),
        "date": date,
        "time": time_s,
        "hour": hour,
        "when_uploaded": choose_first(obs.get("created_at"), obs.get("updated_at")),
        "country": "",
        "country_code": "",
        "location": place_guess,
        "latitude": safe_float(lat),
        "longitude": safe_float(lon),
        "remarks": choose_first(obs.get("description"), obs.get("description_html")),
        "license": choose_first(sound.get("license_code"), obs.get("license_code")),
        "audio_quality": "research-grade",
        "verification": "iNaturalist research-grade",
        "audio_url": audio_url,
        "observation_url": choose_first(obs.get("uri"), f"https://www.inaturalist.org/observations/{obs_id}" if obs_id else None),
        "file_name": safe_filename_from_url(str(audio_url), f"{source_id}.audio"),
        "raw_json": json.dumps({"observation": obs, "sound": sound}, ensure_ascii=False),
    }


def fetch_inaturalist_metadata() -> pd.DataFrame:
    observations, amphibia_taxon_id = inat_fetch_amphibia_observations()

    if MAX_RECORDS_PER_SOURCE_SPECIES:
        observations = observations[:MAX_RECORDS_PER_SOURCE_SPECIES]

    rows = []
    for obs in observations:
        for sound in obs.get("sounds", []) or []:
            row = normalise_inat_sound(obs, sound, amphibia_taxon_id)
            if row["audio_url"]:
                rows.append(row)

    pd.DataFrame([{
        "scope": DISCOVERY_SCOPE,
        "inat_amphibia_taxon_id": amphibia_taxon_id,
    }]).to_csv(RAW_META_DIR / "inat_taxon_lookup.csv", index=False)

    df = pd.DataFrame(rows)
    out = RAW_META_DIR / "inaturalist_raw_metadata.csv"
    df.to_csv(out, index=False)
    print(f"Saved {len(df):,} iNaturalist metadata rows to {out}")
    return df


inat_meta = fetch_inaturalist_metadata()
inat_meta.head()

Saved 6,881 iNaturalist metadata rows to AnuraSetEurope_rebuild/raw/metadata/inaturalist_raw_metadata.csv


,source,source_id,source_record_id,sound_id,query_name,repository_taxon_id,latin,repository_latin,common_name,recordist,...,latitude,longitude,remarks,license,audio_quality,verification,audio_url,observation_url,file_name,raw_json
0,inat,INAT854583_1804,854583,1804,Anura,1319807,Pelophylax ridibundus,Pelophylax ridibundus,Marsh Frog,chrislex,...,47.323436,8.524827,Photographing a pond I walked into this frogs'...,cc-by-nc-nd,research-grade,iNaturalist research-grade,https://static.inaturalist.org/sounds/1804.mp3...,http://www.inaturalist.org/observations/854583,1804.mp3,"{""observation"": {""quality_grade"": ""research"", ..."
1,inat,INAT1213468_2179,1213468,2179,Anura,25616,Rana italica,Rana italica,Italian Stream Frog,finrod,...,42.071572,12.642558,"about 20 males heard (and recorded) calling, q...",cc-by-nc,research-grade,iNaturalist research-grade,https://static.inaturalist.org/sounds/2179.mp3...,http://www.inaturalist.org/observations/1213468,2179.mp3,"{""observation"": {""quality_grade"": ""research"", ..."
2,inat,INAT1217371_2269,1217371,2269,Anura,25616,Rana italica,Rana italica,Italian Stream Frog,danieleseglie,...,42.089282,12.728515,3-4 calling males; recorded with hydrophone (c...,cc-by-nc,research-grade,iNaturalist research-grade,https://static.inaturalist.org/sounds/2269.mp3...,http://www.inaturalist.org/observations/1217371,2269.mp3,"{""observation"": {""quality_grade"": ""research"", ..."
3,inat,INAT1240903_2283,1240903,2283,Anura,25664,Rana latastei,Rana latastei,Italian Agile Frog,danieleseglie,...,44.981043,7.781730,Calling Males; T.air: 1.3; T.water: 6.8. Recor...,cc-by-nc,research-grade,iNaturalist research-grade,https://static.inaturalist.org/sounds/2283.mp3...,http://www.inaturalist.org/observations/1240903,2283.mp3,"{""observation"": {""quality_grade"": ""research"", ..."
4,inat,INAT1240904_8208,1240904,8208,Anura,25669,Rana dalmatina,Rana dalmatina,Agile Frog,danieleseglie,...,44.872090,7.678660,"2/3 calling males; T.air: 2,0; T.water: 6,3",cc-by-nc,research-grade,iNaturalist research-grade,https://static.inaturalist.org/sounds/8208.mp3...,http://www.inaturalist.org/observations/1240904,8208.mp3,"{""observation"": {""quality_grade"": ""research"", ..."


### 4.3 Observation.org through GBIF

This section queries the Observation.org GBIF dataset for sound media. GBIF exposes occurrence data and multimedia links, but the availability of direct audio URLs can vary by publisher settings and licence. Always inspect the downloaded metadata and licence terms before redistributing files.

In [10]:
def gbif_match_taxon_key(name: str, rank: Optional[str] = None) -> Optional[int]:
    url = "https://api.gbif.org/v1/species/match"
    params = {
        "name": name,
        "kingdom": "Animalia",
    }
    if rank:
        params["rank"] = rank.upper()

    data = request_json(url, params=params)
    key = choose_first(data.get("usageKey"), data.get("acceptedUsageKey"), data.get("speciesKey"))
    return int(key) if key is not None else None


def gbif_fetch_obsorg_amphibia_occurrences() -> Tuple[List[dict], int]:
    amphibia_class_key = gbif_match_taxon_key("Amphibia", rank="CLASS")
    if not amphibia_class_key:
        raise ValueError("Could not resolve GBIF classKey for Amphibia.")

    url = "https://api.gbif.org/v1/occurrence/search"
    offset = 0
    limit = 300
    records = []

    while True:
        params = {
            "datasetKey": GBIF_OBSORG_DATASET_KEY,
            "classKey": amphibia_class_key,
            "continent": "EUROPE",
            "mediaType": "Sound",
            "limit": limit,
            "offset": offset,
        }

        data = request_json(url, params=params)
        results = data.get("results", [])
        records.extend(results)

        if data.get("endOfRecords", True) or not results:
            break
        offset += limit

    return records, amphibia_class_key


def normalise_gbif_obsorg_media(occ: dict, media: dict, amphibia_class_key: int) -> dict:
    gbif_id = choose_first(occ.get("gbifID"), occ.get("key"))
    media_id = choose_first(media.get("identifier"), media.get("references"), media.get("title"))
    media_hash = md5_string(str(media_id))[:10]
    source_id = f"OBSORG{gbif_id}_{media_hash}"

    event_date = choose_first(occ.get("eventDate"), occ.get("verbatimEventDate"), occ.get("dateIdentified"))
    date, time_s, hour = parse_date_time(event_date)
    audio_url = choose_first(media.get("identifier"), media.get("references"))

    repository_latin = choose_first(
        occ.get("species"),
        occ.get("acceptedScientificName"),
        occ.get("scientificName"),
    )

    return {
        "source": "observation_org_gbif",
        "source_id": source_id,
        "source_record_id": gbif_id,
        "sound_id": media_hash,
        "query_name": DISCOVERY_SCOPE,
        "repository_taxon_id": choose_first(
            occ.get("speciesKey"),
            occ.get("acceptedTaxonKey"),
            occ.get("taxonKey"),
            amphibia_class_key,
        ),
        "latin": repository_latin,
        "repository_latin": repository_latin,
        "recordist": choose_first(occ.get("recordedBy"), occ.get("identifiedBy"), occ.get("rightsHolder")),
        "observer": choose_first(occ.get("recordedBy"), occ.get("identifiedBy")),
        "date": date,
        "time": time_s,
        "hour": hour,
        "when_uploaded": choose_first(occ.get("lastInterpreted"), occ.get("modified")),
        "country": choose_first(occ.get("country"), occ.get("countryCode")),
        "country_code": choose_first(occ.get("countryCode"), ""),
        "location": choose_first(occ.get("locality"), occ.get("stateProvince"), occ.get("municipality")),
        "latitude": safe_float(occ.get("decimalLatitude")),
        "longitude": safe_float(occ.get("decimalLongitude")),
        "altitude": safe_float(choose_first(occ.get("elevation"), occ.get("minimumElevationInMeters"))),
        "remarks": choose_first(occ.get("occurrenceRemarks"), occ.get("identificationRemarks")),
        "license": choose_first(media.get("license"), occ.get("license"), occ.get("rights")),
        "audio_quality": choose_first(occ.get("degreeOfEstablishment"), ""),
        "verification": "GBIF interpreted Observation.org occurrence",
        "audio_url": audio_url,
        "observation_url": choose_first(occ.get("references"), f"https://www.gbif.org/occurrence/{gbif_id}" if gbif_id else None),
        "file_name": safe_filename_from_url(str(audio_url), f"{source_id}.audio"),
        "raw_json": json.dumps({"occurrence": occ, "media": media}, ensure_ascii=False),
    }


def fetch_observation_org_gbif_metadata() -> pd.DataFrame:
    occurrences, amphibia_class_key = gbif_fetch_obsorg_amphibia_occurrences()

    if MAX_RECORDS_PER_SOURCE_SPECIES:
        occurrences = occurrences[:MAX_RECORDS_PER_SOURCE_SPECIES]

    rows = []

    for occ in occurrences:
        for media in occ.get("media", []) or []:
            media_type = normalise_text(choose_first(media.get("type"), media.get("format"))).lower()

            if "sound" not in media_type and "audio" not in media_type:
                url = str(choose_first(media.get("identifier"), media.get("references"), "")).lower()
                if not re.search(r"\.(wav|mp3|m4a|ogg|flac)(\?|$)", url):
                    continue

            row = normalise_gbif_obsorg_media(occ, media, amphibia_class_key)
            if row["audio_url"]:
                rows.append(row)

    pd.DataFrame([{
        "scope": DISCOVERY_SCOPE,
        "gbif_amphibia_class_key": amphibia_class_key,
    }]).to_csv(RAW_META_DIR / "gbif_taxon_lookup.csv", index=False)

    df = pd.DataFrame(rows)
    out = RAW_META_DIR / "observation_org_gbif_raw_metadata.csv"
    df.to_csv(out, index=False)
    print(f"Saved {len(df):,} Observation.org/GBIF metadata rows to {out}")
    return df


obsorg_meta = fetch_observation_org_gbif_metadata()
obsorg_meta.head()

Saved 1,835 Observation.org/GBIF metadata rows to AnuraSetEurope_rebuild/raw/metadata/observation_org_gbif_raw_metadata.csv


,source,source_id,source_record_id,sound_id,query_name,repository_taxon_id,latin,repository_latin,recordist,observer,...,longitude,altitude,remarks,license,audio_quality,verification,audio_url,observation_url,file_name,raw_json
0,observation_org_gbif,OBSORG6182501202_1496e20df4,6182501202,1496e20df4,Anura,2424293,Pelodytes punctatus,Pelodytes punctatus,User 266191,User 266191,...,2.487571,NaN,None,http://creativecommons.org/licenses/by-nc-nd/4.0/,None,GBIF interpreted Observation.org occurrence,https://observation.org/sounds/346069.mp3,https://www.gbif.org/occurrence/6182501202,346069.mp3,"{""occurrence"": {""key"": 6182501202, ""datasetKey..."
1,observation_org_gbif,OBSORG6183346176_70652e7663,6183346176,70652e7663,Anura,2424293,Pelodytes punctatus,Pelodytes punctatus,User 266191,User 266191,...,2.482057,NaN,None,http://creativecommons.org/licenses/by-nc-nd/4.0/,None,GBIF interpreted Observation.org occurrence,https://observation.org/sounds/346071.mp3,https://www.gbif.org/occurrence/6183346176,346071.mp3,"{""occurrence"": {""key"": 6183346176, ""datasetKey..."
2,observation_org_gbif,OBSORG6228551379_7162f936a1,6228551379,7162f936a1,Anura,5217160,Bufo bufo,Bufo bufo,User 794466,User 794466,...,5.700000,NaN,None,http://creativecommons.org/licenses/by-nc-nd/4.0/,None,GBIF interpreted Observation.org occurrence,https://observation.org/sounds/354703.wav,https://www.gbif.org/occurrence/6228551379,354703.wav,"{""occurrence"": {""key"": 6228551379, ""datasetKey..."
3,observation_org_gbif,OBSORG6169112625_02bcc407a5,6169112625,02bcc407a5,Anura,2426789,Rana arvalis,Rana arvalis,User 213743,User 213743,...,4.550000,NaN,None,http://creativecommons.org/licenses/by-nc-nd/4.0/,None,GBIF interpreted Observation.org occurrence,https://observation.org/sounds/346218.wav,https://www.gbif.org/occurrence/6169112625,346218.wav,"{""occurrence"": {""key"": 6169112625, ""datasetKey..."
4,observation_org_gbif,OBSORG6169221586_a4a76b41ac,6169221586,a4a76b41ac,Anura,2426760,Rana dalmatina,Rana dalmatina,User 926434,User 926434,...,9.842568,NaN,None,http://creativecommons.org/licenses/by-nc-nd/4.0/,None,GBIF interpreted Observation.org occurrence,https://observation.org/sounds/347490.wav,https://www.gbif.org/occurrence/6169221586,347490.wav,"{""occurrence"": {""key"": 6169221586, ""datasetKey..."


## 5. Combine and harmonise source metadata

The harmonised manifest keeps one row per candidate sound file before audio curation. Additional source-specific fields are retained in `raw_json`.

In [11]:
PUBLIC_METADATA_FIELDS = [
    ("anuraset id", "anuraset_id", "Stable dataset identifier: species, source, and source record ID."),
    ("source", "source", "Originating repository: xc, inat, or observation_org_gbif."),
    ("source id", "source_id", "Unique recording or observation ID within the source repository."),
    ("sound id", "sound_id", "Sound/media identifier where available."),
    ("genus", "genus", "Resolved genus."),
    ("species", "species", "Resolved species epithet."),
    ("subspecies", "subspecies", "Resolved subspecies epithet where available."),
    ("latin", "latin", "Standardized binomial species name."),
    ("common name", "common_name", "English common name where available."),
    ("recordist", "recordist", "Recorder, observer, or contributor name."),
    ("date", "date", "Recording or observation date."),
    ("time", "time", "Recording or observation time where available."),
    ("when uploaded", "when_uploaded", "Repository upload or interpretation timestamp."),
    ("country", "country", "Country of recording."),
    ("location", "location", "Locality or site description."),
    ("latitude", "latitude", "Latitude coordinate."),
    ("longitude", "longitude", "Longitude coordinate."),
    ("remarks", "remarks", "Source-provided notes or description."),
    ("rec method", "rec_method", "Recording/observation method where available."),
    ("voc type", "voc_type", "Vocalisation type where available."),
    ("sampling rate", "sampling_rate", "Sampling rate of the exported WAV file in Hz."),
    ("length", "length", "Duration of the curated audio file in seconds."),
    ("observation url", "observation_url", "URL of the original observation or record."),
    ("audio url", "audio_url", "URL of the original audio file."),
    ("file name", "file_name", "Curated audio filename."),
    ("license", "license", "Source-provided licence."),
    ("audio quality", "audio_quality", "Source-provided audio quality or quality grade."),
    ("verification", "verification", "Source-specific verification status."),
    ("file path", "file_path", "Local path of the curated exported audio file."),
    ("retrieval path", "retrieval_path", "Local path used for the initially downloaded source file."),
    ("audio hash", "audio_hash", "Content-derived waveform hash."),
    ("group key", "group_key", "Grouped acoustic-session key used for splitting."),
    ("split", "split", "Dataset split: train, validation, or test."),
]

STANDARD_COLUMNS = [
    "candidate_id",
    "anuraset_id",
    "source",
    "source_id",
    "source_record_id",
    "sound_id",
    "query_name",
    "repository_taxon_id",
    "repository_latin",
    "genus",
    "species",
    "subspecies",
    "latin",
    "common_name",
    "recordist",
    "observer",
    "date",
    "time",
    "hour",
    "when_uploaded",
    "country",
    "country_code",
    "location",
    "latitude",
    "longitude",
    "remarks",
    "rec_method",
    "voc_type",
    "sampling_rate",
    "native_sampling_rate",
    "length",
    "observation_url",
    "audio_url",
    "file_name",
    "license",
    "audio_quality",
    "verification",
    "file_path",
    "retrieval_path",
    "audio_hash",
    "group_key",
    "split",
    "raw_json",
]


def is_blank(x) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    return str(x).strip() == "" or str(x).strip().lower() in {"nan", "none", "null"}


def clean_value(x) -> str:
    return "" if is_blank(x) else str(x).strip()


def first_value(*values) -> str:
    for v in values:
        if not is_blank(v):
            return clean_value(v)
    return ""


def parse_raw_json(x) -> dict:
    if is_blank(x):
        return {}
    if isinstance(x, dict):
        return x
    try:
        return json.loads(x)
    except Exception:
        return {}


def parse_xc_length_to_seconds(x):
    if is_blank(x):
        return ""

    s = clean_value(x)

    if re.fullmatch(r"\d+(\.\d+)?", s):
        return float(s)

    parts = s.split(":")
    try:
        parts = [float(p) for p in parts]
    except Exception:
        return ""

    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]

    return ""


def standardise_taxon_from_name(name) -> dict:
    name = clean_value(name)
    if not name:
        return {"genus": "", "species": "", "subspecies": "", "latin": ""}

    name = re.sub(r"\([^)]*\)", "", name)
    name = re.sub(r"\s+", " ", name).strip()

    parts = name.split(" ")
    if len(parts) < 2:
        return {"genus": "", "species": "", "subspecies": "", "latin": ""}

    genus = parts[0]
    species = parts[1]

    if species.lower() in {"sp", "sp.", "spp", "spp.", "cf", "cf.", "aff", "aff.", "complex"}:
        return {"genus": "", "species": "", "subspecies": "", "latin": ""}

    subspecies = ""
    if len(parts) >= 3 and parts[2].islower():
        subspecies = parts[2]

    return {
        "genus": genus,
        "species": species,
        "subspecies": subspecies,
        "latin": f"{genus} {species}",
    }


def make_anuraset_id(row: pd.Series) -> str:
    return f"ASE_{slugify(str(row['latin']))}_{str(row['source']).upper()}_{slugify(str(row['source_id']))}"


def build_retrieval_path(row: pd.Series) -> str:
    ext = Path(str(row["file_name"])).suffix.lower()
    if not ext:
        ext = ".audio"

    fname = f"{row['source']}_{slugify(str(row['source_id']))}{ext}"
    species_dir = slugify(str(row["latin"])) if not is_blank(row["latin"]) else "unresolved_taxon"

    return str(RAW_AUDIO_DIR / str(row["source"]) / species_dir / fname)


def extract_source_specific_fields(row: pd.Series) -> dict:
    source = clean_value(row.get("source"))
    raw = parse_raw_json(row.get("raw_json"))

    out = {}

    if source == "xc":
        genus = first_value(raw.get("gen"))
        species = first_value(raw.get("sp"))
        subspecies = first_value(raw.get("ssp"))

        if genus and species:
            out["genus"] = genus
            out["species"] = species
            out["subspecies"] = subspecies
            out["latin"] = f"{genus} {species}"

        out["common_name"] = first_value(row.get("common_name"), raw.get("en"))
        out["rec_method"] = first_value(raw.get("method"))
        out["voc_type"] = first_value(raw.get("type"))
        out["native_sampling_rate"] = first_value(raw.get("smp"))
        out["sampling_rate"] = first_value(raw.get("smp"))
        out["length"] = parse_xc_length_to_seconds(raw.get("length"))
        out["file_name"] = first_value(raw.get("file-name"), row.get("file_name"))
        out["verification"] = first_value(row.get("verification"), "xeno-canto")

    elif source == "observation_org_gbif":
        occ = raw.get("occurrence", raw)
        media = raw.get("media", {})

        genus = first_value(occ.get("genus"), occ.get("genericName"))
        species_epithet = first_value(occ.get("specificEpithet"))

        if genus and species_epithet:
            taxon = standardise_taxon_from_name(f"{genus} {species_epithet}")
        else:
            taxon = standardise_taxon_from_name(
                first_value(
                    occ.get("species"),
                    occ.get("acceptedScientificName"),
                    occ.get("scientificName"),
                    row.get("latin"),
                    row.get("repository_latin"),
                )
            )

        out.update(taxon)
        out["common_name"] = first_value(row.get("common_name"), occ.get("vernacularName"))
        out["rec_method"] = first_value(occ.get("samplingProtocol"))
        out["time"] = first_value(occ.get("eventTime"), row.get("time"))
        out["license"] = first_value(media.get("license"), occ.get("license"), row.get("license"))
        out["audio_url"] = first_value(media.get("accessURI"), media.get("identifier"), row.get("audio_url"))
        out["verification"] = first_value(
            occ.get("identificationVerificationStatus"),
            row.get("verification"),
            "GBIF interpreted Observation.org occurrence",
        )

    elif source == "inat":
        obs = raw.get("observation", raw)
        sound = raw.get("sound", {})
        taxon = obs.get("taxon", {}) or {}

        rank = clean_value(taxon.get("rank")).lower()

        if rank in {"species", "subspecies"}:
            out.update(standardise_taxon_from_name(taxon.get("name")))
        else:
            out.update({"genus": "", "species": "", "subspecies": "", "latin": ""})

        out["common_name"] = first_value(
            row.get("common_name"),
            taxon.get("preferred_common_name"),
            obs.get("species_guess"),
        )
        out["license"] = first_value(sound.get("license_code"), obs.get("license_code"), row.get("license"))
        out["verification"] = first_value(row.get("verification"), "iNaturalist research-grade")

    return out


def harmonise_metadata(dfs: Iterable[pd.DataFrame]) -> pd.DataFrame:
    dfs = [df for df in dfs if df is not None and len(df) > 0]

    if not dfs:
        raise ValueError("No source metadata found. Check source-download cells first.")

    df = pd.concat(dfs, ignore_index=True, sort=False)
    df = df.drop_duplicates(subset=["source", "source_id", "audio_url"], keep="first").copy()

    rows = []

    for _, row in df.iterrows():
        out = {col: "" for col in STANDARD_COLUMNS}

        for col in STANDARD_COLUMNS:
            if col in row.index:
                out[col] = clean_value(row.get(col))

        source_specific = extract_source_specific_fields(row)

        for key, value in source_specific.items():
            if key in out and not is_blank(value):
                out[key] = clean_value(value)

        if is_blank(out["latin"]):
            taxon = standardise_taxon_from_name(first_value(row.get("latin"), row.get("repository_latin")))
            out["genus"] = taxon["genus"]
            out["species"] = taxon["species"]
            out["subspecies"] = taxon["subspecies"]
            out["latin"] = taxon["latin"]

        if is_blank(out["genus"]) or is_blank(out["species"]):
            taxon = standardise_taxon_from_name(out["latin"])
            out["genus"] = first_value(out["genus"], taxon["genus"])
            out["species"] = first_value(out["species"], taxon["species"])
            out["subspecies"] = first_value(out["subspecies"], taxon["subspecies"])
            out["latin"] = first_value(out["latin"], taxon["latin"])

        out["retrieval_path"] = build_retrieval_path(pd.Series(out))
        out["anuraset_id"] = make_anuraset_id(pd.Series(out))

        rows.append(out)

    out_df = pd.DataFrame(rows)

    if INCLUDE_ONLY_BINOMIAL_SPECIES:
        before = len(out_df)
        out_df = out_df[
            out_df["latin"].fillna("").astype(str).str.match(r"^[A-Z][A-Za-z-]+ [a-z][a-z-]+$")
        ].copy()
        print(f"Dropped non-species-level records: {before - len(out_df):,}")

    out_df = out_df.reset_index(drop=True)
    out_df["candidate_id"] = [f"ASE_CAND_{i:07d}" for i in range(1, len(out_df) + 1)]

    out_df["recordist_norm"] = out_df["recordist"].fillna("").astype(str).str.lower().str.strip()
    out_df["lat_round_5"] = pd.to_numeric(out_df["latitude"], errors="coerce").round(5)
    out_df["lon_round_5"] = pd.to_numeric(out_df["longitude"], errors="coerce").round(5)
    out_df["lat_round_3"] = pd.to_numeric(out_df["latitude"], errors="coerce").round(3)
    out_df["lon_round_3"] = pd.to_numeric(out_df["longitude"], errors="coerce").round(3)

    ordered_cols = STANDARD_COLUMNS + ["recordist_norm", "lat_round_5", "lon_round_5", "lat_round_3", "lon_round_3"]
    out_df = out_df[ordered_cols]

    out = RAW_META_DIR / "candidate_manifest_before_download.csv"
    out_df.to_csv(out, index=False)
    print(f"Saved {len(out_df):,} candidate rows to {out}")

    field_availability = (
        out_df[[internal for _, internal, _ in PUBLIC_METADATA_FIELDS]]
        .replace("", pd.NA)
        .notna()
        .mean()
        .mul(100)
        .round(1)
        .reset_index()
        .rename(columns={"index": "field", 0: "percent_populated"})
    )

    availability_out = RAW_META_DIR / "candidate_manifest_field_availability.csv"
    field_availability.to_csv(availability_out, index=False)
    print(f"Saved field availability report to {availability_out}")

    display(out_df.groupby("source").size().sort_values(ascending=False).rename("n_records").reset_index())
    display(field_availability)

    return out_df


candidate_manifest = harmonise_metadata([xc_meta, inat_meta, obsorg_meta])
candidate_manifest.head()

Dropped non-species-level records: 1,093
Saved 9,850 candidate rows to AnuraSetEurope_rebuild/raw/metadata/candidate_manifest_before_download.csv
Saved field availability report to AnuraSetEurope_rebuild/raw/metadata/candidate_manifest_field_availability.csv


,source,n_records
0,inat,6522
1,xc,1894
2,observation_org_gbif,1434


,field,percent_populated
0,anuraset_id,100.0
1,source,100.0
2,source_id,100.0
3,sound_id,100.0
4,genus,100.0
5,species,100.0
6,subspecies,5.8
7,latin,100.0
8,common_name,99.8
9,recordist,100.0


,candidate_id,anuraset_id,source,source_id,source_record_id,sound_id,query_name,repository_taxon_id,repository_latin,genus,...,retrieval_path,audio_hash,group_key,split,raw_json,recordist_norm,lat_round_5,lon_round_5,lat_round_3,lon_round_3
0,ASE_CAND_0000001,ASE_Pelophylax_bedriagae_XC_XC1155635,xc,XC1155635,1155635,1155635,Anura,,Pelophylax bedriagae,Pelophylax,...,AnuraSetEurope_rebuild/raw/audio/xc/Pelophylax...,,,,"{""id"": ""1155635"", ""gen"": ""Pelophylax"", ""sp"": ""...",jagoda hanuszewicz,40.0900,29.5600,40.090,29.560
1,ASE_CAND_0000002,ASE_Pelophylax_bedriagae_XC_XC1155631,xc,XC1155631,1155631,1155631,Anura,,Pelophylax bedriagae,Pelophylax,...,AnuraSetEurope_rebuild/raw/audio/xc/Pelophylax...,,,,"{""id"": ""1155631"", ""gen"": ""Pelophylax"", ""sp"": ""...",jagoda hanuszewicz,39.5152,29.3816,39.515,29.382
2,ASE_CAND_0000003,ASE_Pelophylax_bedriagae_XC_XC1155624,xc,XC1155624,1155624,1155624,Anura,,Pelophylax bedriagae,Pelophylax,...,AnuraSetEurope_rebuild/raw/audio/xc/Pelophylax...,,,,"{""id"": ""1155624"", ""gen"": ""Pelophylax"", ""sp"": ""...",jagoda hanuszewicz,39.4100,30.1900,39.410,30.190
3,ASE_CAND_0000004,ASE_Pelophylax_bedriagae_XC_XC1155621,xc,XC1155621,1155621,1155621,Anura,,Pelophylax bedriagae,Pelophylax,...,AnuraSetEurope_rebuild/raw/audio/xc/Pelophylax...,,,,"{""id"": ""1155621"", ""gen"": ""Pelophylax"", ""sp"": ""...",jagoda hanuszewicz,40.2100,32.2500,40.210,32.250
4,ASE_CAND_0000005,ASE_Pelophylax_bedriagae_XC_XC1155620,xc,XC1155620,1155620,1155620,Anura,,Pelophylax bedriagae,Pelophylax,...,AnuraSetEurope_rebuild/raw/audio/xc/Pelophylax...,,,,"{""id"": ""1155620"", ""gen"": ""Pelophylax"", ""sp"": ""...",jagoda hanuszewicz,40.2100,32.2500,40.210,32.250


## 6. Download audio

This cell downloads all candidate files into `raw/audio/`. Rerunning the cell skips files already present on disk.

In [12]:
def download_candidate_audio(manifest: pd.DataFrame) -> pd.DataFrame:
    manifest = manifest.copy()
    statuses = []
    if not DOWNLOAD_AUDIO:
        manifest["download_ok"] = False
        print("DOWNLOAD_AUDIO=False; skipping downloads.")
        return manifest

    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Downloading audio"):
        out_path = Path(row["retrieval_path"])
        ok = download_binary(str(row["audio_url"]), out_path)
        statuses.append(ok)
    manifest["download_ok"] = statuses
    out = RAW_META_DIR / "candidate_manifest_after_download.csv"
    manifest.to_csv(out, index=False)
    print(f"Downloaded/available: {manifest['download_ok'].sum():,}/{len(manifest):,}")
    print(f"Saved {out}")
    return manifest

candidate_manifest = download_candidate_audio(candidate_manifest)
candidate_manifest["download_ok"].value_counts(dropna=False)

Download failed: https://static.inaturalist.org/sounds/698478.m4a?1684164347 -> AnuraSetEurope_rebuild/raw/audio/inat/Pelophylax_lessonae/inat_INAT161813886_698478.m4a (503 Server Error: Service Unavailable for url: https://static.inaturalist.org/sounds/698478.m4a?1684164347)
Downloaded/available: 9,848/9,850
Saved AnuraSetEurope_rebuild/raw/metadata/candidate_manifest_after_download.csv


download_ok
True     9848
False       2
Name: count, dtype: int64

*Note: The final number of recordings decreased after filtering out species with not enough support (less than 5 audio files) and removing duplicates with MD5 checksums. We then added recordings shared with us directly from the FonoZoo library. Resulting raw audio files and corresponding metadata can be found [here](https://doi.org/10.5281/zenodo.21282595).*